# ⚙️ Préparation des Données & Pipeline de Preprocessing

**Objectif :** Transformer les enseignements tirés de l'EDA en un pipeline de préparation de données propre, étanche (sans *Data Leakage*) et directement exploitable par nos futurs modèles de Machine Learning.

---

### 🎯 Étapes du Preprocessing
1. **Sélection des variables (*Feature Selection*) :** Suppression argumentée des colonnes inutiles (constantes, identifiants, bruit aléatoire, risques éthiques et fortes colinéarités).
2. **Ingénierie des variables (*Feature Engineering*) :** Création de ratios métiers d'ancienneté et transformation du salaire.
3. **Séparation Train / Test stratifiée :** Règle d'or anti-*Data Leakage* appliquée avant tout encodage ou mise à l'échelle.
4. **Pipeline d'Encodage & Mise à l'échelle :** *ColumnTransformer* combinant *OneHotEncoder*, *OrdinalEncoder* et *RobustScaler*.
5. **Gestion du déséquilibre de classe :** Définition de la stratégie de pondération (*Class Weights*) et SMOTE.
6. **Exportation des données prêtes :** Sauvegarde des jeux traités dans `data/processed/`.

In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
# Modules Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler

# Configuration visuelle
sns.set_theme(style="whitegrid", palette="muted")

In [2]:
# Chargement des données brutes
df = pd.read_csv("../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv")

print(f"Dimensions initiales : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head(3)

Dimensions initiales : 1470 lignes, 35 colonnes


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0


## 1. Sélection des Variables (*Feature Selection*)

Sur la base des constats établis lors de l'EDA, nous filtrons **10 colonnes** qui ne doivent pas entrer dans le modèle prédictif :

1. **Identifiant technique & Constantes (aucune variance) :**
   - `EmployeeNumber` : simple index technique.
   - `EmployeeCount` (constante = 1), `StandardHours` (constante = 80), `Over18` (constante = 'Y').
2. **Critère éthique & légal :**
   - `Gender` : exclue pour des raisons de fairness. Il s'agit d'une caractéristique sensible dont l'utilisation dans une décision RH automatisée peut introduire ou reproduire des biais discriminatoires. Son association avec l'attrition est par ailleurs faible dans cet échantillon.
3. **Variables de pur bruit aléatoire :**
   - `DailyRate`, `HourlyRate`, `MonthlyRate` : Variables à faible valeur analytique apparente : distributions quasi uniformes et absence d'association notable avec l'attrition dans l'EDA. Elles sont exclues du modèle initial.
4. **Multicolinéarité sévère (Redondances) :**
   Pour le modèle initial, nous conservons `MonthlyIncome` afin de représenter la dimension de rémunération avec une variable continue, et excluons `JobLevel` pour limiter la redondance.
   - `PerformanceRating` : corrélé à **0.77** avec `PercentSalaryHike` et ne contenant que deux notes (3 et 4). Nous conservons l'augmentation en pourcentage.

In [5]:
# Définition des variables exclues pour le modèle initial
cols_to_drop = [
    # 1. Identifiant et variables sans variance (constantes)
    "EmployeeNumber",
    "EmployeeCount",
    "StandardHours",
    "Over18",
    # 2. Caractéristique sensible (considérations éthiques et de fairness)
    "Gender",
    # 3. Variables à faible valeur analytique apparente
    "DailyRate",
    "HourlyRate",
    "MonthlyRate",
    # 4. Multicolinéarité / Limitation de la redondance
    "JobLevel",
    "PerformanceRating",
]

target_col = "Attrition"

# Application de la sélection
df_clean = df.drop(columns=cols_to_drop).copy()

print(
    f"Dimensions après sélection : {df_clean.shape[0]} observations, {df_clean.shape[1]} variables"
)
print(
    f"-> 1 variable cible ('{target_col}') et {df_clean.shape[1] - 1} variables explicatives retenues."
)

Dimensions après sélection : 1470 observations, 25 variables
-> 1 variable cible ('Attrition') et 24 variables explicatives retenues.


## 2. Ingénierie des Variables (*Feature Engineering*)

Afin de fournir un meilleur signal prédictif aux algorithmes, nous créons des variables dérivées fondées sur la dynamique de carrière et la distribution des revenus :

1. **Ratios d'ancienneté et de mobilité interne :**
   - `YearsWithCurrManager_Ratio` = $\frac{\text{YearsWithCurrManager}}{\text{YearsAtCompany} + 1}$  
     *Interprétation :* Mesure la stabilité relationnelle avec l'encadrement par rapport au parcours dans l'entreprise (un ajout de $+1$ au dénominateur neutralise le risque de division par zéro pour les nouveaux arrivants).
   - `YearsSinceLastPromotion_Ratio` = $\frac{\text{YearsSinceLastPromotion}}{\text{YearsAtCompany} + 1}$  
     *Interprétation :* Évalue la proportion du temps passé sans évolution hiérarchique au sein de l'organisation (indicateur de stagnation relative).
   - `CompanyTenure_Ratio` = $\frac{\text{YearsAtCompany}}{\text{TotalWorkingYears} + 1}$  
     *Interprétation :* Mesure le degré de fidélité historique en distinguant les collaborateurs ayant réalisé l'essentiel de leur carrière dans l'entreprise de ceux recrutés récemment après une expérience externe substantielle.

2. **Normalisation de l'asymétrie salariale :**
   - `MonthlyIncome_Log` = $\log(1 + \text{MonthlyIncome})$  
     *Interprétation :* Comme constaté lors de l'EDA, `MonthlyIncome` présente une asymétrie positive marquée ($\text{Skewness} = 1.37$). La transformation logarithmique stabilise la dispersion et ramène la distribution vers une quasi-symétrie ($\text{Skewness} \approx 0.29$), facilitant ainsi la convergence des estimateurs linéaires et basés sur des métriques de distance.

In [6]:
# 1. Ratios de trajectoire professionnelle (avec lissage +1 au dénominateur)
df_clean["YearsWithCurrManager_Ratio"] = df_clean["YearsWithCurrManager"] / (
    df_clean["YearsAtCompany"] + 1
)
df_clean["YearsSinceLastPromotion_Ratio"] = df_clean[
    "YearsSinceLastPromotion"
] / (df_clean["YearsAtCompany"] + 1)
df_clean["CompanyTenure_Ratio"] = df_clean["YearsAtCompany"] / (
    df_clean["TotalWorkingYears"] + 1
)

# 2. Transformation logarithmique du revenu mensuel
df_clean["MonthlyIncome_Log"] = np.log1p(df_clean["MonthlyIncome"])

# Nous retirons le MonthlyIncome d'origine pour éviter la colinéarité avec sa version transformée
df_clean = df_clean.drop(columns=["MonthlyIncome"])

# Contrôle de la qualité des transformations
print(
    f"Nombre de valeurs manquantes après feature engineering : {df_clean.isna().sum().sum()}"
)
print(f"Nouvelles dimensions du dataset : {df_clean.shape}")
df_clean[[
    "YearsWithCurrManager_Ratio",
    "YearsSinceLastPromotion_Ratio",
    "CompanyTenure_Ratio",
    "MonthlyIncome_Log",
]].head(3)

Nombre de valeurs manquantes après feature engineering : 0
Nouvelles dimensions du dataset : (1470, 28)


,YearsWithCurrManager_Ratio,YearsSinceLastPromotion_Ratio,CompanyTenure_Ratio,MonthlyIncome_Log
0,0.714286,0.000000,0.666667,8.698514
1,0.636364,0.090909,0.909091,8.543056
2,0.000000,0.000000,0.000000,7.645398


## 3. Séparation Train / Test avec Stratification (Anti-Data Leakage)

Afin d'évaluer nos futurs modèles de façon rigoureuse et non biaisée, nous appliquons deux principes méthodologiques fondamentaux :

1. **Étanchéité méthodologique (Prévention du *Data Leakage*) :**  
   Le découpage entre jeu d'entraînement (*Train*) et jeu de test (*Test*) est effectué **avant** tout ajustement d'encodeur ou de mise à l'échelle. Les paramètres de transformation seront appris exclusivement sur le Train set (`fit_transform`) puis appliqués au Test set (`transform`).

2. **Stratification sur la variable cible (`stratify=y`) :**  
   Compte tenu de la faible proportion de la classe minoritaire (16.1% de départs), un découpage purement aléatoire risquerait de créer une distorsion entre les deux ensembles. La stratification assure une conservation stricte des proportions de départs dans le Train (80%) et dans le Test (20%).

In [7]:
# 1. Séparation des features explicatives (X) et de la variable cible (y)
X = df_clean.drop(columns=["Attrition"])

# Encodage binaire de la cible : 1 pour 'Yes' (départ), 0 pour 'No' (maintien)
y = (df_clean["Attrition"] == "Yes").astype(int)

# 2. Découpage stratifié 80% Train / 20% Test (random_state fixé pour la reproductibilité)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Contrôle des volumes et de la représentativité
print("--- Répartition des sous-ensembles ---")
print(
    f"Jeu d'entraînement (Train) : {X_train.shape[0]} observations ({X_train.shape[1]} features)"
)
print(
    f"Jeu de test (Test)         : {X_test.shape[0]} observations ({X_test.shape[1]} features)"
)

print("\n--- Contrôle de la stratification (Taux d'Attrition) ---")
print(
    f"Ensemble complet : {y.mean()*100:.2f}% ({y.sum()} départs / {len(y)})"
)
print(
    f"Train set        : {y_train.mean()*100:.2f}% ({y_train.sum()} départs / {len(y_train)})"
)
print(
    f"Test set         : {y_test.mean()*100:.2f}% ({y_test.sum()} départs / {len(y_test)})"
)

--- Répartition des sous-ensembles ---
Jeu d'entraînement (Train) : 1176 observations (27 features)
Jeu de test (Test)         : 294 observations (27 features)

--- Contrôle de la stratification (Taux d'Attrition) ---
Ensemble complet : 16.12% (237 départs / 1470)
Train set        : 16.16% (190 départs / 1176)
Test set         : 15.99% (47 départs / 294)


## 4. Pipeline d'Encodage et de Mise à l'Échelle (*ColumnTransformer*)

Afin d'uniformiser le traitement des données tout en garantissant une étanchéité totale, nous encapsulons nos transformations dans un `ColumnTransformer` :

1. **Variables Catégorielles Nominales (One-Hot Encoding) :**  
   - Traitement par `OneHotEncoder(drop='first', sparse_output=False)` : chaque modalité devient une colonne binaire (0 ou 1). L'exclusion de la première modalité (`drop='first'`) neutralise le piège de la variable muette (*dummy variable trap*) et prévient la dépendance linéaire parfaite.

2. **Variables Numériques et Ordinales (RobustScaler) :**  
   - Traitement par `RobustScaler()` : compte tenu de l'étalement et des valeurs extrêmes observés lors de l'EDA, nous privilégions une normalisation fondée sur la médiane et l'écart interquartile (IQR), moins vulnérable aux outliers que la standardisation par moyenne/écart-type (`StandardScaler`).

3. **Application rigoureuse :**  
   - Le transformateur apprend exclusivement les références statistiques sur le sous-ensemble d'apprentissage (`X_train`), puis projette le sous-ensemble de test (`X_test`) dans ce même espace sans réajustement.

In [8]:
# 1. Identification automatique des colonnes par nature de traitement
nominal_cols = X_train.select_dtypes(
    include=["object", "string"]
).columns.tolist()
numeric_and_ordinal_cols = [
    c for c in X_train.columns if c not in nominal_cols
]

print(f"Nombre de variables nominales à encoder (OHE)   : {len(nominal_cols)}")
print(
    f"Nombre de variables numériques/ordinales à scaler : {len(numeric_and_ordinal_cols)}"
)

# 2. Définition du ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            "nominal",
            OneHotEncoder(
                drop="first", sparse_output=False, handle_unknown="ignore"
            ),
            nominal_cols,
        ),
        ("num_ord", RobustScaler(), numeric_and_ordinal_cols),
    ],
    verbose_feature_names_out=False,  # Permet d'obtenir des noms de colonnes propres sans préfixe
)

# 3. Ajustement sur X_train et transformation des deux sous-ensembles
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 4. Reconstruction sous forme de DataFrames pandas documentés
feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(
    X_train_processed, columns=feature_names, index=X_train.index
)
X_test_df = pd.DataFrame(
    X_test_processed, columns=feature_names, index=X_test.index
)

# Contrôle du résultat
print("\n--- Bilan du Preprocessing ---")
print(f"Dimensions de X_train_df : {X_train_df.shape}")
print(f"Dimensions de X_test_df  : {X_test_df.shape}")
print(
    f"Contrôle d'absence de valeurs manquantes (Train) : {X_train_df.isna().sum().sum()} NaN"
)
print(
    f"Contrôle d'absence de valeurs manquantes (Test)  : {X_test_df.isna().sum().sum()} NaN"
)

# Aperçu des 3 premières observations traitées
X_train_df.head(3)

Nombre de variables nominales à encoder (OHE)   : 6
Nombre de variables numériques/ordinales à scaler : 21

--- Bilan du Preprocessing ---
Dimensions de X_train_df : (1176, 41)
Dimensions de X_test_df  : (294, 41)
Contrôle d'absence de valeurs manquantes (Train) : 0 NaN
Contrôle d'absence de valeurs manquantes (Test)  : 0 NaN


,BusinessTravel_Travel_Frequently,BusinessTravel_Travel_Rarely,Department_Research & Development,Department_Sales,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree,JobRole_Human Resources,...,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,YearsWithCurrManager_Ratio,YearsSinceLastPromotion_Ratio,CompanyTenure_Ratio,MonthlyIncome_Log
1194,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-1.0,0.0,-0.285714,-0.2,0.000000,-0.2,0.000000,0.250000,-1.177332,1.105822
128,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-1.0,0.0,-0.428571,-0.4,0.333333,-0.4,-0.500000,1.222222,-0.304604,-0.652510
810,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.000000,1.2,1.000000,1.2,0.576923,0.384615,-0.304604,1.190979


## 5. Traitement du Déséquilibre de Classe (*Class Imbalance*)

Avec un ratio d'environ 1 départ pour 5 maintiens dans le Train set (16.2% de classe minoritaire), un modèle standard risquerait de privilégier la classe majoritaire au détriment de la détection des départs réels. Nous préparons deux stratégies complémentaires :

1. **Pondération des classes (*Class Weights*) :**  
   Calcul des poids inversement proportionnels aux fréquences de classe : chaque observation positive (`Attrition = 1`) recevra un coefficient d'environ **3.09** contre **0.60** pour la classe majoritaire. Cette approche pénalise davantage les faux négatifs directement au niveau de la fonction de coût des algorithmes.

2. **Sur-échantillonnage synthétique (*SMOTE*) :**  
   Génération d'observations synthétiques via la méthode *Synthetic Minority Over-sampling Technique*. Conformément aux exigences méthodologiques, cette opération est appliquée **strictement sur l'ensemble d'entraînement (`X_train`)**, maintenant l'ensemble de test (`X_test`) dans sa distribution empirique d'origine.

In [9]:
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight

# 1. Calcul des poids théoriques équilibrés (Class Weights)
classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced", classes=classes, y=y_train
)
class_weights_dict = {
    int(cls): round(float(w), 3) for cls, w in zip(classes, weights)
}

print("--- Poids de pondération calculés (Class Weights) ---")
print(f"Poids classe 0 (Maintien) : {class_weights_dict[0]}")
print(
    f"Poids classe 1 (Départ)   : {class_weights_dict[1]} (pénalité x{class_weights_dict[1]/class_weights_dict[0]:.1f})"
)

# 2. Application de SMOTE exclusivement sur le Train set
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_df, y_train)

print("\n--- Évolution des effectifs dans le Train Set ---")
print(
    f"Train initial : {dict(y_train.value_counts())} (Total : {len(y_train)})"
)
print(
    f"Train SMOTE   : {dict(y_train_smote.value_counts())} (Total : {len(y_train_smote)})"
)
print(
    f"Test (inchangé) : {dict(y_test.value_counts())} (Total : {len(y_test)})"
)

--- Poids de pondération calculés (Class Weights) ---
Poids classe 0 (Maintien) : 0.596
Poids classe 1 (Départ)   : 3.095 (pénalité x5.2)

--- Évolution des effectifs dans le Train Set ---
Train initial : {0: np.int64(986), 1: np.int64(190)} (Total : 1176)
Train SMOTE   : {0: np.int64(986), 1: np.int64(986)} (Total : 1972)
Test (inchangé) : {0: np.int64(247), 1: np.int64(47)} (Total : 294)


## 6. Exportation des Jeux de Données Prétraités

Les jeux de données transformés sont exportés au format CSV dans le répertoire `../data/processed/` :
- `X_train.csv` et `y_train.csv` : données d'entraînement standard (avec pondération possible via `class_weights`).
- `X_train_smote.csv` et `y_train_smote.csv` : données d'entraînement rééquilibrées par SMOTE.
- `X_test.csv` et `y_test.csv` : données de test inviolées servant à l'évaluation finale indépendante.

In [10]:
# Définition du chemin de sortie
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

# 1. Export des jeux standards
X_train_df.to_csv(f"{output_dir}/X_train.csv", index=False)
X_test_df.to_csv(f"{output_dir}/X_test.csv", index=False)
y_train.to_csv(f"{output_dir}/y_train.csv", index=False)
y_test.to_csv(f"{output_dir}/y_test.csv", index=False)

# 2. Export des jeux rééquilibrés SMOTE
X_train_smote.to_csv(f"{output_dir}/X_train_smote.csv", index=False)
y_train_smote.to_csv(f"{output_dir}/y_train_smote.csv", index=False)

# 3. Contrôle des fichiers créés sur le disque
print("Fichiers exportés avec succès dans data/processed/ :")
for file in sorted(os.listdir(output_dir)):
  file_path = os.path.join(output_dir, file)
  size_kb = os.path.getsize(file_path) / 1024
  print(f"  ✓ {file:22s} ({size_kb:6.1f} Ko)")

Fichiers exportés avec succès dans data/processed/ :
  ✓ X_test.csv             (  89.2 Ko)
  ✓ X_train.csv            ( 353.0 Ko)
  ✓ X_train_smote.csv      ( 723.4 Ko)
  ✓ y_test.csv             (   0.6 Ko)
  ✓ y_train.csv            (   2.3 Ko)
  ✓ y_train_smote.csv      (   3.9 Ko)


### 🚀 Bilan du Preprocessing & Prochaines Étapes

Le pipeline de prétraitement est désormais complet et méthodologiquement clos :
1. **Élimination des biais :** Exclusion des constantes, des variables sensibles (`Gender`) et du bruit.
2. **Gain de signal :** Intégration de ratios métiers et réduction de l'asymétrie salariale via `MonthlyIncome_Log`.
3. **Étanchéité totale :** Aucune fuite de données entre l'entraînement et l'évaluation.
4. **Matériel prêt pour la modélisation :** 41 variables explicatives standardisées prêtes à alimenter les modèles comparatifs (Régression Logistique, Random Forest, LightGBM/XGBoost) dans le prochain notebook `3-Modeling.ipynb`.